In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

In [2]:
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA_DIR / "PhiUSIIL_Preprocessed.csv"
)

with open(
    RESULTS_DIR / "feature_config.json",
    "r"
) as f:
    config = json.load(f)

full_features = config["full_features"]
url_only_features = config["url_only_features"]

print("Jumlah sampel:", len(df))
print("Jumlah full features:", len(full_features))
print("Jumlah URL-only features:", len(url_only_features))

df.head()

Jumlah sampel: 235370
Jumlah full features: 47
Jumlah URL-only features: 19


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [3]:
feature_audit = pd.DataFrame({
    "Feature": url_only_features,
    "Available": [
        feature in df.columns
        for feature in url_only_features
    ],
    "Data Type": [
        str(df[feature].dtype)
        if feature in df.columns else "Missing"
        for feature in url_only_features
    ],
    "Missing Values": [
        int(df[feature].isna().sum())
        if feature in df.columns else None
        for feature in url_only_features
    ]
})

display(feature_audit)

,Feature,Available,Data Type,Missing Values
0,URLLength,True,int64,0
1,DomainLength,True,int64,0
2,IsDomainIP,True,int64,0
3,TLDLength,True,int64,0
4,NoOfSubDomain,True,int64,0
5,HasObfuscation,True,int64,0
6,NoOfObfuscatedChar,True,int64,0
7,ObfuscationRatio,True,float64,0
8,NoOfLettersInURL,True,int64,0
9,LetterRatioInURL,True,float64,0


In [4]:
assert df["URL"].is_unique, \
    "Masih terdapat URL duplikat."

assert not df["Domain"].isna().any(), \
    "Terdapat domain kosong."

assert set(full_features).issubset(df.columns), \
    "Ada full features yang tidak ditemukan."

assert set(url_only_features).issubset(df.columns), \
    "Ada URL-only features yang tidak ditemukan."

assert set(url_only_features).issubset(full_features), \
    "URL-only features bukan subset full features."

assert df[full_features].select_dtypes(
    exclude=np.number
).empty, "Terdapat fitur nonnumerik."

assert np.isfinite(
    df[full_features].to_numpy(dtype=float)
).all(), "Terdapat NaN atau infinity pada fitur."

print("Validasi dataset dan fitur berhasil.")

Validasi dataset dan fitur berhasil.


In [5]:
X = df[full_features]

y = df["label"].map({
    0: 1,
    1: 0
})

groups = df["Domain"].astype(str)

assert not y.isna().any()

print("Jumlah sampel:", len(X))
print("Jumlah domain unik:", groups.nunique())

print("\nDistribusi target:")
print(y.value_counts())

Jumlah sampel: 235370
Jumlah domain unik: 220086

Distribusi target:
label
0    134850
1    100520
Name: count, dtype: int64


In [6]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(
        X,
        y,
        groups=groups
    )
)

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

print(
    "Training percentage:",
    round(len(train_idx) / len(df) * 100, 2)
)

print(
    "Testing percentage:",
    round(len(test_idx) / len(df) * 100, 2)
)

Training samples: 165149
Testing samples: 70221
Training percentage: 70.17
Testing percentage: 29.83


In [7]:
train_domains = set(groups.iloc[train_idx])
test_domains = set(groups.iloc[test_idx])

overlap = train_domains.intersection(test_domains)

print("Jumlah domain training:", len(train_domains))
print("Jumlah domain testing:", len(test_domains))
print("Jumlah domain overlap:", len(overlap))

assert len(overlap) == 0, \
    "Domain overlap ditemukan!"

print("\nDomain-grouped split berhasil.")

Jumlah domain training: 154060
Jumlah domain testing: 66026
Jumlah domain overlap: 0

Domain-grouped split berhasil.


In [8]:
train_labels = y.iloc[train_idx]
test_labels = y.iloc[test_idx]

class_distribution = pd.DataFrame({
    "Training": train_labels.value_counts(
        normalize=True
    ) * 100,

    "Testing": test_labels.value_counts(
        normalize=True
    ) * 100
})

class_distribution.index = (
    class_distribution.index.map({
        0: "Legitimate",
        1: "Phishing"
    })
)

display(class_distribution.round(2))

assert set(train_labels.unique()) == {0, 1}
assert set(test_labels.unique()) == {0, 1}

print("Kedua kelas tersedia pada training dan testing.")

,Training,Testing
label,,
Legitimate,57.18,57.57
Phishing,42.82,42.43


Kedua kelas tersedia pada training dan testing.


In [9]:
random_split = np.load(
    RESULTS_DIR / "baseline_split.npz"
)

random_train_idx = random_split["train_idx"]
random_test_idx = random_split["test_idx"]

random_train_domains = set(
    groups.iloc[random_train_idx]
)

random_test_domains = set(
    groups.iloc[random_test_idx]
)

random_overlap = (
    random_train_domains
    .intersection(random_test_domains)
)

print("Random Split Overlap:", len(random_overlap))
print("Domain-Grouped Split Overlap:", len(overlap))

Random Split Overlap: 2633
Domain-Grouped Split Overlap: 0


In [10]:
split_comparison = pd.DataFrame({
    "Split Strategy": [
        "Random Split",
        "Domain-Grouped Split"
    ],
    "Training Samples": [
        len(random_train_idx),
        len(train_idx)
    ],
    "Testing Samples": [
        len(random_test_idx),
        len(test_idx)
    ],
    "Domain Overlap": [
        len(random_overlap),
        len(overlap)
    ]
})

display(split_comparison)

,Split Strategy,Training Samples,Testing Samples,Domain Overlap
0,Random Split,164759,70611,2633
1,Domain-Grouped Split,165149,70221,0


In [11]:
np.savez(
    RESULTS_DIR / "domain_grouped_split.npz",
    train_idx=train_idx,
    test_idx=test_idx
)

split_comparison.to_csv(
    RESULTS_DIR / "split_comparison.csv",
    index=False
)

class_distribution.to_csv(
    RESULTS_DIR / "grouped_class_distribution.csv"
)

print("Hasil Domain-Grouped Split berhasil disimpan.")

Hasil Domain-Grouped Split berhasil disimpan.


In [12]:
output_files = [
    "domain_grouped_split.npz",
    "split_comparison.csv",
    "grouped_class_distribution.csv"
]

for file in output_files:

    path = RESULTS_DIR / file

    print(
        file,
        "->",
        "Tersedia" if path.exists() else "Tidak ditemukan"
    )

domain_grouped_split.npz -> Tersedia
split_comparison.csv -> Tersedia
grouped_class_distribution.csv -> Tersedia


Berdasarkan hasil pembagian dataset PhiUSIIL menggunakan GroupShuffleSplit, diperoleh 165.149 data pelatihan dan 70.221 data pengujian, dengan 154.060 domain unik pada data pelatihan dan 66.026 domain unik pada data pengujian. Hasil pemeriksaan menunjukkan bahwa tidak terdapat domain yang tumpang tindih antara kedua kelompok (domain overlap = 0). Sebagai pembanding, strategi random split menghasilkan 2.633 domain yang muncul pada data pelatihan dan pengujian. Hasil ini menunjukkan bahwa domain-grouped split berhasil menerapkan pemisahan berdasarkan domain sehingga dapat digunakan untuk mengevaluasi performa model pada domain yang belum ditemukan selama pelatihan. Namun, pengaruh strategi tersebut terhadap performa deteksi phishing masih perlu dibuktikan melalui eksperimen klasifikasi.